# Scraping Ikman.lk - First Look

Ikman uses a modern JavaScript framework (React/Next.js). Instead of rendering the car data as raw HTML out of the gate, they often inject a massive JSON object into a `<script>` tag called `window.initialData`.

This is fantastic news for scraping! Instead of writing fragile BeautifulSoup selectors to find classes like `class="title-1234"`, we can simply use a regular expression to grab that single JSON string and convert it straight into a Pandas DataFrame.

In [ ]:
import requests
import re
import json
import pandas as pd

# Target URL
url = 'https://ikman.lk/en/ads/sri-lanka/cars'

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
}

# Fetch the page
response = requests.get(url, headers=headers)
print(f"Status Code: {response.status_code}")
if response.status_code == 200:
    # Extract window.initialData using Regex
    # re.DOTALL is important because the JSON might contain newlines
    match = re.search(r'window\.initialData\s*=\s*({.*?});</script>', response.text, re.DOTALL)
    
    if match:
        raw_json_str = match.group(1)
        try:
            data = json.loads(raw_json_str)
            print("Successfully parsed the injected JSON object!")
        except json.JSONDecodeError as e:
            print(f"Failed to parse JSON: {e}")
            data = None
    else:
        print("Pattern 'window.initialData' not found in the HTML.")
        data = None

Status Code: 200
Pattern 'window.initialData' not found in the HTML.


Now let's navigate the dictionary and pull out the actual ads:

In [ ]:
if data is not None:
    try:
        # Drilling down into the specific nested dictionaries where the ads live
        ads_list = data['serp']['ads']['data']['ads']
        
        print(f"Found {len(ads_list)} ads on the first page.")
        
        # Convert directly to DataFrame
        df = pd.DataFrame(ads_list)
        
        print("\nAvailable keys/columns:", list(df.columns))
        
        # Show the most useful columns
        display_cols = ['title', 'price', 'location', 'details', 'timeStamp', 'imgUrl']
        # Filter out ones that might be missing just in case
        display_cols = [c for c in display_cols if c in df.columns]
        
        display(df[display_cols].head())
        
    except KeyError as e:
        print(f"JSON structure has changed. Target key missing: {e}")